# 基于M3E特征的零样本酒店评论情感识别

In [3]:
# 运行准备：按第5.2.2节读取并划分酒店评论数据，复用示例代码5.66和5.67准备M3E测试嵌入
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import sentence_transformers as sentrans

data_file = "../pybook-data/ch5/ChnSentiCorp_htl_all.csv"
df = pd.read_csv(data_file).dropna(subset=["review"])
sents = df["review"].tolist()
y = df["label"].tolist()

out_dir = "output"
train_file = f"{out_dir}/ChnSentiCorp_train.csv"
test_file = f"{out_dir}/ChnSentiCorp_test.csv"
if os.path.exists(train_file) and os.path.exists(test_file):
    print(f'Files exist at \"{train_file}\" and \"{test_file}\"')
    df_train, df_test = pd.read_csv(train_file), pd.read_csv(test_file)
    sents_test = df_test["review"].tolist()
    y_test = df_test["label"].tolist()
else:
    os.makedirs(out_dir, exist_ok=True)
    sents_train, sents_test, y_train, y_test = train_test_split(sents, y, test_size=0.2, random_state=1)
    print(f'训练样本数量：{len(sents_train)}')
    print(f'测试样本数量：{len(sents_test)}')
    pd.DataFrame({'review': sents_train, 'label': y_train}).to_csv(train_file, index=False)
    pd.DataFrame({'review': sents_test, 'label': y_test}).to_csv(test_file, index=False)

model = sentrans.SentenceTransformer("./fm/m3e-small")
x_test = model.encode(sents_test)

Files exist at "output/ChnSentiCorp_train.csv" and "output/ChnSentiCorp_test.csv"


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 2034.48it/s]


In [4]:
label_text = ['负面酒店评论是指顾客在入住体验后，对酒店服务、设施、卫生、环境等方面表达不满或失望的评论。这类评论通常包含对具体问题的描述，如房间脏乱、服务态度差、噪音扰民、设施故障、价格与体验不符等，旨在提醒其他潜在住客或促使酒店改进。', '正面酒店评论是顾客在入住后对酒店表示满意或赞赏的评价。这类评论通常提及服务热情周到、房间整洁舒适、设施齐全先进、位置便利、性价比高或超出预期等优点，旨在推荐给其他潜在住客并肯定酒店的表现。']
label_emb = model.encode(label_text)
scores = sentrans.util.cos_sim(x_test, label_emb)
pred = np.argmax(scores, axis=1)
pred = pred.cpu().numpy()
report = classification_report(y_test, pred, digits=3)
print(report)

              precision    recall  f1-score   support

           0      0.595     0.941     0.729       476
           1      0.965     0.717     0.823      1077

    accuracy                          0.786      1553
   macro avg      0.780     0.829     0.776      1553
weighted avg      0.852     0.786     0.794      1553

